In [ ]:
import sys
import pandas as pd
import numpy as np
import pysam

from tqdm.notebook import tqdm
from preprocessing_tools import *

sys.path.append('code')
tqdm.pandas()

In [ ]:
contexts = pd.read_csv('data/COSMIC/COSMIC_v3.4_SBS_GRCh38.txt', usecols=['Type'], sep='\t')
assembly = pd.read_json('data/ncbi_genome/ncbi_dataset/data/sequence_report.jsonl', lines=True)
chr_id = dict(zip(assembly['ucscStyleName'], assembly['refseqAccession']))

genome = pysam.FastaFile('data/ncbi_genome/ncbi_dataset/data/GCF_000001405.13/GCF_000001405.13_GRCh37_genomic.fna')

In [3]:
def reverse_complement(seq):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement[base] for base in reversed(seq))

def get_sbs(row):
    ref = genome.fetch(chr_id['chr' + str(row['chr'])], row['start'] - 2, row['start'] + 1).upper()
    if('N' in ref):
        return None
    
    alt = row['alt']

    if(ref[1] != row['ref']):
        print('Reference mismatch:', row['chr'], row['start'], row['ref'], ref)

    if(ref[1] == 'A' or ref[1] == 'G'):
        ref = reverse_complement(ref)
        alt = reverse_complement(alt)

    return ref[0] + '[' + ref[1] + '>' + alt + ']' + ref[2]

In [4]:
df = pd.read_csv('../data/funnell/DLP/SNV/snv_counts.csv.gz', usecols=['chr', 'start', 'ref', 'alt', 'cell_id', 'patient'])

In [8]:
df['sbs'] = df.progress_apply(get_sbs, axis=1)

  0%|          | 0/31385551 [00:00<?, ?it/s]

In [9]:
df

,chr,start,ref,alt,cell_id,patient,sbs
0,1,2901334,A,T,SA604X6XB01972-A96177C-R55-C31,SA604,A[T>A]G
1,1,4724917,C,G,SA604X6XB01972-A96177C-R55-C31,SA604,A[C>G]C
2,1,5404076,A,T,SA604X6XB01972-A96177C-R55-C31,SA604,G[T>A]A
3,1,6303195,T,G,SA604X6XB01972-A96177C-R55-C31,SA604,A[T>G]C
4,1,11585442,G,T,SA604X6XB01972-A96177C-R55-C31,SA604,C[C>A]C
...,...,...,...,...,...,...,...
31385546,6,54625196,T,C,AT135-A118389B-R65-C67,SA1292,A[T>C]G
31385547,6,70850297,T,G,AT135-A118389B-R65-C67,SA1292,A[T>G]C
31385548,8,92983198,G,T,AT135-A118389B-R65-C67,SA1292,G[C>A]C
31385549,X,85749714,T,A,AT135-A118389B-R65-C67,SA1292,A[T>A]C


In [10]:
for patient_id in tqdm(df['patient'].unique()):
    df_patient = df[df['patient'] == patient_id]
    table = pd.crosstab(df_patient['cell_id'], df_patient['sbs'])
    table = table.reindex(columns=contexts['Type'], fill_value=0) # Ensure all contexts are present as columns
    table.to_csv(f'../data/funnell_tables/{patient_id}_sbs_counts.csv')

  0%|          | 0/35 [00:00<?, ?it/s]